# CS781 Phase 7 — Notebook P7_03: re-fuse saved model scores with the newest lexical scores (CPU, minutes)

Every P7_02 experiment saved its dense score matrices for dev and test (`scores.npz`: variants mean, maxsim, both,
ctx_max, ctx_maxsim). Fusing them with a better lexical system needs **no GPU**: this notebook loads every attached
experiment, chooses (dense variant, λ, time penalty γ) on dev exactly like P7_02 —
`z(dense) + λ·z(lexical without time rule) − γ·[candidate later than query]` — and writes new submissions.
It also builds an **ensemble** of all trained models of this account (mean of their z-normalised best dense scores),
as an extra (not one of the four required) submission.

**Kaggle settings**: CPU, Internet not needed. **Inputs**: (1) the newest **P7_01 output** (v3), (2) this
account's latest **P7_02 output** (it contains the earlier rounds' `scores.npz` as well).
**Output**: `/kaggle/working/cs781_p1/p7/fused/<experiment>/submission_hybrid.zip` + `p7/fused/fused_results.csv`.

In [ ]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

In [ ]:
import pandas as pd, zipfile
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
try: display
except NameError: display = print

META = load_json(find_artifact("p7/data/meta.json")); assert META, "attach the P7_01 output"
print("lexical system:", META["best_lexical_system"], "| weights:", META.get("lexpp_weights"), "| time rule γ:", META.get("year_gamma"), "tol:", META.get("year_tol"))
IDS = {s: load_json(find_artifact(f"p7/data/{s}_ids.json")) for s in ("dev", "test")}
GOLD = {"dev": load_json(find_artifact("p7/data/dev_gold.json"))}
LEX = {s: np.load(find_artifact(f"p7/data/{s}_lex.npz")) for s in ("dev", "test")}
HAS_LEXPP = "best_noyear" in LEX["dev"].files
YEAR_GAMMAS = (0.0, 1.0, 3.0, 100.0) if HAS_LEXPP and META.get("year_gamma", 0) > 0 else (0.0,)
LAMBDAS = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0)
def canon(x):
    s = str(x).strip(); return (s.lstrip("0") or "0") if s.isdigit() else s

def z(S):
    S = S.astype(np.float32); fin = np.isfinite(S); m = np.where(fin, S, np.nan)
    mu = np.nanmean(m, 1, keepdims=True); sd = np.nanstd(m, 1, keepdims=True) + 1e-6
    out = (S - mu) / sd; out[~fin] = -np.inf; return out
def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1)
    return np.take_along_axis(idx, o, 1)
def f1_at(S, K=7):
    order = topk(S, K); g = GOLD["dev"]["gold"]; ng = sum(GOLD["dev"]["n_gold"])
    hits = sum(len(set(row.tolist()) & set(gg)) for row, gg in zip(order, g))
    p = hits / (K * len(order)); r = hits / max(1, ng)
    return {"F1@7": 2 * p * r / max(1e-12, p + r), "P@7": p, "R@7": r}
SELF = {}
for s in ("dev", "test"):
    pos = {canon(x): i for i, x in enumerate(IDS[s]["candidates"])}
    SELF[s] = [(qi, pos[canon(q)]) for qi, q in enumerate(IDS[s]["queries"]) if canon(q) in pos]
def mask_self(S, s):
    S = S.astype(np.float32).copy()
    for qi, ci in SELF[s]: S[qi, ci] = -np.inf
    return S
LEXZ = {s: z(LEX[s]["best_noyear"] if HAS_LEXPP else LEX[s]["best"]) for s in ("dev", "test")}
LATE = {s: (LEX[s]["late"].astype(np.float32) if HAS_LEXPP else 0.0) for s in ("dev", "test")}
def hybrid(Sz, s, lam, g):
    H = Sz if lam == 0 else Sz + lam * LEXZ[s]
    return H - g * LATE[s] if g else H
lex_dev = f1_at(LEX["dev"]["best"]); print(f"lexical alone on dev: F1@7 {lex_dev['F1@7']:.4f} (P {lex_dev['P@7']:.4f}, R {lex_dev['R@7']:.4f})")

def write_submission(S, path):
    top = topk(S, 7); cids, qids = IDS["test"]["candidates"], IDS["test"]["queries"]; pred = {}
    for qi, q in enumerate(qids):
        ids = [cids[j] for j in top[qi]]
        assert len(ids) == len(set(ids)) == min(7, len(cids)) and canon(q) not in {canon(x) for x in ids}
        pred[q] = ids
    path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf: zf.writestr("prediction.json", json.dumps(pred, indent=1))

## Attached experiments → best fusion per experiment

In [ ]:
roots = [os.environ["CS781_P7_INPUTS"]] if os.environ.get("CS781_P7_INPUTS") else (["/kaggle/input"] if IN_KAGGLE else [])
roots.append(str(ROOT))
EXPS = {}
for r in roots:
    for f in sorted(glob.glob(f"{r}/**/p7/exps/*/scores.npz", recursive=True)):
        name = Path(f).parent.name; z_ = np.load(f)
        if any(k.startswith("test_") for k in z_.files) and name not in EXPS: EXPS[name] = z_
print("experiments with test scores:", sorted(EXPS))

rows, BEST_DENSE = [], {}
FUSED = ROOT / "p7" / "fused"
for name, sc in sorted(EXPS.items()):
    variants = [k[4:] for k in sc.files if k.startswith("dev_") and f"test_{k[4:]}" in sc.files]
    Zd = {v: z(mask_self(sc[f"dev_{v}"], "dev")) for v in variants}
    dense = {v: f1_at(Zd[v])["F1@7"] for v in variants}
    bv = max(dense, key=dense.get); BEST_DENSE[name] = (bv, dense[bv])
    best = None
    for v in variants:
        for lam in LAMBDAS:
            for g in YEAR_GAMMAS:
                r = f1_at(hybrid(Zd[v], "dev", lam, g))
                if best is None or r["F1@7"] > best[3]["F1@7"]: best = (v, lam, g, r)
    v, lam, g, r = best
    write_submission(hybrid(z(mask_self(sc[f"test_{v}"], "test")), "test", lam, g), FUSED / name / "submission_hybrid.zip")
    rows.append({"name": name, "best dense": f"{bv} {dense[bv]:.4f}", "dev F1@7 hybrid": r["F1@7"], "P@7": r["P@7"], "R@7": r["R@7"], "fusion": f"{v} λ={lam} γ={g}"})

## Ensemble of this account's trained models (extra submission)
Mean of the z-normalised best dense score of every trained experiment (frozen `zs_*` and duplicates of the same
trained weights — `*_rs` — are left out), then fused with the lexical scores the same way.

In [ ]:
members = [n for n in EXPS if not n.startswith("zs_") and not n.endswith("_rs") and n in BEST_DENSE]
if len(members) >= 2:
    Ed = np.mean([z(mask_self(EXPS[n][f"dev_{BEST_DENSE[n][0]}"], "dev")) for n in members], 0)
    Et = np.mean([z(mask_self(EXPS[n][f"test_{BEST_DENSE[n][0]}"], "test")) for n in members], 0)
    Ed, Et = z(Ed), z(Et); best = None
    for lam in LAMBDAS:
        for g in YEAR_GAMMAS:
            r = f1_at(hybrid(Ed, "dev", lam, g))
            if best is None or r["F1@7"] > best[2]["F1@7"]: best = (lam, g, r)
    lam, g, r = best
    write_submission(hybrid(Et, "test", lam, g), FUSED / "ensemble" / "submission_hybrid.zip")
    rows.append({"name": f"ensemble of {len(members)}", "best dense": f"{f1_at(Ed)['F1@7']:.4f}", "dev F1@7 hybrid": r["F1@7"], "P@7": r["P@7"], "R@7": r["R@7"], "fusion": f"λ={lam} γ={g}"})
    print("ensemble members:", members)
write_submission(LEX["test"]["best"], FUSED / "lexical_only" / "submission.zip")
rows.append({"name": "lexical only (no Phase-1 model)", "best dense": "-", "dev F1@7 hybrid": lex_dev["F1@7"], "P@7": lex_dev["P@7"], "R@7": lex_dev["R@7"], "fusion": META["best_lexical_system"]})
df = pd.DataFrame(rows).sort_values("dev F1@7 hybrid", ascending=False); display(df.round(4))
df.to_csv(FUSED / "fused_results.csv", index=False)
print("submissions written under", FUSED)